# Sampling: Temperature, Top-k, and Nucleus

Topic 14 asked for the single most probable string and found that it is short, bland, and often empty. Sampling asks for
something else: a string drawn *from* the model, so that every string turns up as often as the model says it should.
This notebook makes three points:
- **sampling is exact** where search is not, and the two differ only by noise;
- **temperature and truncation bias the draw**, by amounts we can compute exactly;
- **neither can add knowledge the model lacks**: they sharpen what the model believes, whether it is right or wrong.

The engines are #14's: hand-built toy trees (exact by enumeration), Topic 3's Kneser–Ney trigram, and a small causal
decoder, sampled 8 times on each of 188 held-out MD&A prompts under 25 settings. The finance case study adds an order-5
Kneser–Ney model as a fact model and scores its answers against the synthetic corpus's knowledge base.

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "sampling_temperature_nucleus.py").exists()
                       else here / "notebooks" / "sampling-temperature-nucleus"))
import sampling_temperature_nucleus as st

S = st.summary()
print(S["n_prompts"], "prompts ×", st.N_PER_PROMPT, "draws =", S["draws_per_setting"], "draws per setting")
print(f"trigram per-word NLL: mode {S['trigram_mode_nll']:.3f}, pure samples {S['trigram_temperature1_nll']:.3f}, "
      f"held-out text {S['trigram_heldout_nll']:.3f}")

## 1. Sampling is exact; search is not

**Definition 1 (ancestral sampling).** Given a prompt $x$, draw $Y_1 \sim p(\cdot \mid x)$, then
$Y_t \sim p(\cdot \mid x, Y_{\lt t})$, until $\texttt{</s>}$ is drawn.

**Theorem 1 (sampling is exact).**
- **(i)** For every finite string $y$ ending in $\texttt{</s>}$, ancestral sampling returns $y$ with probability
  exactly $p(y \mid x)$.
- **(ii)** It halts with probability $\sum_y p(y \mid x)$. So it halts almost surely if and only if the model is
  consistent (Topic 5).

*Proof.*
- **(i)** The draws are conditionally independent given the prefix. So the probability of drawing
  $y_1, \dots, y_n$ in order is $\prod_t p(y_t \mid x, y_{\lt t}) = p(y \mid x)$, by the chain rule.
- **(ii)** The events "the sampler returns $y$" are disjoint for different finite $y$, and their union is the event
  that it halts. Then (i) gives $P(\text{halt}) = \sum_y p(y \mid x)$. $\square$

Search and sampling look like different algorithms. One fact about Gumbel noise shows that they differ only in what is
added to the scores before taking a maximum.

**Proposition 1 (Gumbel-max).** Let $z \in \mathbb{R}^V$ and let $G_w$ be i.i.d. standard Gumbel, with CDF
$\exp(-e^{-g})$. Then $\arg\max_w (z_w + G_w) \sim \operatorname{softmax}(z)$. The maximum is distributed as
$\mathrm{Gumbel}(\log \sum_w e^{z_w})$ and is independent of the argmax.

*Proof.* Write $Z = \sum_w e^{z_w}$. The density of $z_i + G_i$ at $g$ is $e^{z_i - g}\exp(-e^{z_i - g})$, and
$P(z_j + G_j \le g) = \exp(-e^{z_j - g})$. So
$$P\big(\arg\max = i,\ \max \le m\big) = \int_{-\infty}^{m} e^{z_i - g}\, \exp\!\Big(-\sum_j e^{z_j - g}\Big)\, dg
= \frac{e^{z_i}}{Z}\, \exp\!\big(-Z e^{-m}\big).$$
The right side factors into $\operatorname{softmax}(z)_i$ times the CDF of $\mathrm{Gumbel}(\log Z)$ at $m$.
$\square$

So a draw is the argmax of perturbed log-probabilities. Greedy decoding is the same argmax with the noise set to zero.
Kool, van Hoof, and Welling (2019) push this to whole sequences: they perturb every prefix consistently and run beam
search on the perturbed scores, and the top $k$ strings are then an exact sample without replacement (their Thm 1,
Gumbel-top-$k$).

In [ ]:
st.test_ancestral_sampling_is_exact()
st.test_halting_is_consistency()
st.test_gumbel_max()
lab = st.bs.lab_tree()
p = st.tree_dist(lab)
for y in sorted(p, key=lambda y: -p[y])[:4]:
    print(f"{' '.join(lab.words(y[:-1])):24} p = {p[y]:.4f}")

## 2. Temperature as a Gibbs distribution

**Definition 2 (temperature).** For a next-token distribution with log-probabilities $z = \log p(\cdot \mid h)$ and
$\tau > 0$, $p_\tau(w \mid h) = p(w \mid h)^{1/\tau}/Z_\tau(h) = \operatorname{softmax}(z/\tau)_w$, where
$Z_\tau(h) = \sum_w p(w \mid h)^{1/\tau}$.

**Theorem 2 (temperature is a Gibbs distribution).** For any $z \in \mathbb{R}^V$ and $\tau > 0$:
- **(a)** $p_\tau$ is the unique maximizer over the simplex of $\langle q, z\rangle + \tau H(q)$, and the maximum is
  $\tau \log \sum_w e^{z_w/\tau}$.
- **(b)** $\dfrac{d}{d\tau} H(p_\tau) = \dfrac{\operatorname{Var}_{p_\tau}(z)}{\tau^3} \ge 0$, with equality only when
  $z$ is constant on the support. The expected score $\mathbb{E}_{p_\tau}[z]$ is nonincreasing in $\tau$.
- **(c)** As $\tau \to 0$, $p_\tau$ tends to the uniform distribution on $\arg\max z$. As $\tau \to \infty$, it tends to
  the uniform distribution on the support.

*Proof.*
- **(a)** Write $\mathcal{L} = \log \sum_w e^{z_w/\tau}$. For any $q$,
  $\langle q, z\rangle + \tau H(q) = \tau \mathcal{L} - \tau\, \mathrm{KL}(q \,\|\, p_\tau)$, because
  $\log p_\tau(w) = z_w/\tau - \mathcal{L}$. Gibbs' inequality says the KL term is $\ge 0$, with equality only at
  $q = p_\tau$.
- **(b)** Put $\beta = 1/\tau$ and $A(\beta) = \log \sum_w e^{\beta z_w}$. This is a log-partition function, so
  $A'(\beta) = \mathbb{E}_\beta z$ and $A''(\beta) = \operatorname{Var}_\beta z$ (formalStatistics, exponential
  families, Thm 7.1). Then:
  - $H = A(\beta) - \beta A'(\beta)$, so $dH/d\beta = -\beta\, \operatorname{Var}_\beta z$;
  - $d\beta/d\tau = -1/\tau^2$ turns this into $dH/d\tau = \operatorname{Var}(z)/\tau^3$;
  - $d\,\mathbb{E}_\beta z/d\beta = \operatorname{Var}_\beta z \ge 0$ gives the claim about the expected score.
- **(c)** Divide numerator and denominator by $e^{\beta \max z}$ and let $\beta \to \infty$ or $\beta \to 0$. $\square$

The tempered distributions form an exponential family with sufficient statistic $\log p$ and natural parameter
$\beta$. Annealing walks along that one curve (Jaynes 1957; Kirkpatrick, Gelatt, and Vecchi 1983). Ackley, Hinton, and
Sejnowski (1985) brought the word "temperature" into neural networks.

Temperature also has a picture on Topic 1's log–log plot.

**Proposition 2 (temperature is the Zipf slope).** For any distribution, the least-squares slope of $\log p_\tau$
against $\log(\text{rank})$ is exactly the slope of $\log p$ divided by $\tau$. In particular, a Zipf($s$) row becomes
a Zipf($s/\tau$) row.

*Proof.* Tempering preserves the ranks, and $\log p_\tau = \tfrac{1}{\tau}\log p - \log Z_\tau$. Least squares is
linear in the response and ignores an added constant. $\square$

Sampled *text* is a mixture of many contexts, so its rank–frequency slope only approximately follows $s/\tau$.

In [ ]:
st.test_temperature_is_gibbs()
st.test_entropy_rises_with_temperature()
st.test_temperature_divides_the_zipf_slope()
rows = st.trigram_rows()[0]
lp = rows[st.lab_prompts()[2]]
for tau in (0.5, 1.0, 2.0):
    q = st.temper(lp, tau)
    print(f"tau = {tau}: entropy {st.entropy_rows(q)[0]:.3f} nats, Zipf slope {st.zipf_slope(q):.3f}")

## 3. Local against global temperature

In practice, temperature is applied at every step. That is not the same as tempering the model.

**Theorem 3 (local temperature is not global temperature).** Let
$q_\tau(y) = \prod_t p_\tau(y_t \mid h_t)$ be the per-step tempered sampler, with $h_t$ the prefix before step $t$.
Let $\pi_\tau(y) = p(y)^{1/\tau}/\mathcal{Z}_\tau$ be the tempered sequence distribution, where
$\mathcal{Z}_\tau = \sum_y p(y)^{1/\tau}$, whenever that sum is finite.
- **(a)** $q_\tau(y) = p(y)^{1/\tau}\big/\prod_t Z_\tau(h_t)$. So $q_\tau = \pi_\tau$ if and only if
  $\prod_t Z_\tau(h_t)$ takes the same value for every string in the support. That holds, for example, when every
  next-token distribution is a permutation of one vector and every string has the same length.
- **(b)** Write $\alpha = 1/\tau$ and let $H_\alpha$ be the Rényi entropy of order $\alpha$. Then
  $$\log \frac{q_\tau(y)}{\pi_\tau(y)} = \log \mathcal{Z}_\tau + (\alpha - 1)\sum_t H_\alpha\big(p(\cdot \mid h_t)\big).$$
  For $\tau \lt 1$, local tempering upweights the strings whose prefixes have more accumulated Rényi entropy.
- **(c)** If argmaxes are unique, then as $\tau \to 0$, $q_\tau$ concentrates on greedy's string and $\pi_\tau$ on the
  mode.

*Proof.*
- **(a)** Substitute Definition 2 into the product and collect the numerators, which give $p(y)^{1/\tau}$.
- **(b)** By definition $H_\alpha(p) = \frac{1}{1-\alpha}\log \sum_w p_w^\alpha$, so
  $\log Z_\tau(h) = (1 - \alpha) H_\alpha(p(\cdot \mid h))$. Divide (a) by $\pi_\tau$ and take logs.
- **(c)** At each step $p_\tau(\cdot \mid h)$ tends to the point mass on $h$'s argmax, so on a finite tree $q_\tau$ tends
  to the point mass on the string of step-wise argmaxes, which is greedy's string. Likewise
  $\pi_\tau \propto p(y)^{1/\tau}$ tends to the point mass on $\arg\max_y p(y)$. $\square$

On #14's lab tree, greedy's string is "rose sharply" (0.0864) and the mode is "was flat" (0.238). At $\tau = 0.5$ the
two samplers are already 0.303 apart in total variation.

**Remark (the tempered model need not exist).** For $\tau \gt 1$, $\mathcal{Z}_\tau$ can be infinite. Take Welleck's
stationary model with $k$ words of probability $(1 - \varepsilon)/k$ each and $\texttt{</s>}$ with probability
$\varepsilon$. Then
$$\mathcal{Z}_\tau = \sum_{t \ge 0} k^t \Big(\tfrac{1-\varepsilon}{k}\Big)^{t/\tau} \varepsilon^{1/\tau},$$
a geometric series with ratio $k^{1 - 1/\tau}(1 - \varepsilon)^{1/\tau}$. For $k = 5$ and $\varepsilon = 0.05$ it
diverges once $\tau \ge 1.0319$. The per-step sampler exists at every $\tau$, but "the model at temperature 1.1" does
not.

Caccia et al. (2020, §3) and Shih, Sadigh, and Ermon (2023, §2.1) both observe that per-token temperature differs from
tempering the joint distribution. The normalizer-product form, the Rényi identity, and the divergence remark are what
this section adds.

In [ ]:
st.test_local_temperature_is_not_global()
st.test_tempered_sequence_distribution_can_diverge()
for tau in (1.0, 0.5, 0.1, 0.02):
    loc, glo = st.local_tempered(lab, tau), st.global_tempered(lab, tau)
    g, m = st.bs.greedy(lab, (), max_len=10).tokens, st.bs.brute_force_mode(lab)[1]
    print(f"tau = {tau:4}: local on greedy {loc[g]:.3f}, global on mode {glo[m]:.3f}, TV {st.tv(loc, glo):.3f}")

## 4. Truncation: top-k, nucleus, min-p, η, and typical

**Definition 3 (truncation).** A truncation rule picks a kept set $S(h)$ at each prefix and samples from
$p_S(w \mid h) = p(w \mid h)\,\mathbf{1}[w \in S(h)]/m(h)$, where $m(h) = p(S(h) \mid h)$ is the kept mass. Sort the
probabilities as $p_{(1)} \ge p_{(2)} \ge \cdots$. The rules are:
- **top-$k$:** the $k$ largest (popularized by Fan, Lewis, and Dauphin 2018);
- **nucleus($p$):** the shortest prefix of the sorted order whose mass reaches $p$ (Holtzman et al. 2020);
- **min-$p$($\rho$):** every $w$ with $p_w \ge \rho\, p_{(1)}$ (Nguyen et al. 2025);
- **$\eta$:** every $w$ with $p_w \gt \min(\eta, \sqrt{\eta}\, e^{-H})$, where $H$ is the row's entropy (Hewitt,
  Manning, and Liang 2022);
- **typical:** sort tokens by $|{-\log p_w} - H|$ and keep them until their mass reaches the typical mass
  (Meister et al. 2023, in the greedy form of their erratum).

Ties go to the lower token id.

**Theorem 4 (what truncation costs).**
- **(a)** The nucleus has the fewest tokens of any set with mass $\ge p$, and the most mass among sets of its size.
  Top-$k$ has the most mass among $k$-sets.
- **(b)** Per step, $\mathrm{TV}(p_S, p) = 1 - m$ and $\mathrm{KL}(p_S \,\|\, p) = -\log m$. The reverse divergence
  $\mathrm{KL}(p \,\|\, p_S)$ is infinite when $m \lt 1$.
- **(c)** Per sequence, for the truncated sampler $q$:
  $$\mathrm{KL}(q \,\|\, p) = \mathbb{E}_q\Big[\sum_t -\log m(h_t)\Big], \qquad
  \mathrm{TV}(q, p) \le \mathbb{E}_q\Big[\sum_t \big(1 - m(h_t)\big)\Big],$$
  and the probability that pure sampling never leaves the kept sets is $\mathbb{E}_q\big[\prod_t m(h_t)\big]$.
- **(d)** If every step's kept mass is at most $1 - \varepsilon$, pure sampling stays inside the kept sets for $n$
  steps with probability at most $(1 - \varepsilon)^n$.

*Proof.*
- **(a)** Any $k$ entries sum to at most the $k$ largest. So a set of mass $\ge p$ has at least as many tokens as the
  shortest sorted prefix reaching $p$.
- **(b)** On $S$, $p_S - p = p(1/m - 1) \ge 0$, so $\mathrm{TV} = m(1/m - 1) = 1 - m$. Likewise
  $\mathrm{KL}(p_S \,\|\, p) = \sum_{w \in S} p_S(w) \log(1/m) = -\log m$.
- **(c)**
  - The KL identity is the chain rule for KL: each step contributes $\mathrm{KL}(p_S(\cdot \mid h_t) \,\|\, p(\cdot
    \mid h_t)) = -\log m(h_t)$, averaged over $q$'s prefixes.
  - For TV, couple the two samplers step by step with a maximal coupling. They agree until the first step at which
    they decouple, which happens with probability $1 - m(h_t)$ given agreement so far. A union bound finishes it.
  - For the last identity, $\sum_{y \in \operatorname{supp} q} p(y) = \sum_y q(y)\, p(y)/q(y)$, and
    $p(y)/q(y) = \prod_t m(h_t)$.
- **(d)** Each factor in (c)'s product is at most $1 - \varepsilon$. $\square$

On the trigram, pure sampling leaves the 0.95-nucleus at least once on about a fifth of its continuations.
Estimated directly the share that never leaves is 0.783, and through (c)'s identity under nucleus sampling 0.775.
Holtzman et al.'s "unreliable tail" is not rare over a whole sentence.

In [ ]:
st.test_nucleus_is_minimal()
st.test_truncation_costs_per_step()
st.test_truncation_costs_per_sequence()
st.test_pure_sampling_leaves_the_nucleus()
a = st.avoid_tail("trigram")
print(f"P(pure sampling never leaves nucleus(0.95)): direct {a['direct']:.3f} ± {a['se']:.3f}, "
      f"identity {a['identity']:.3f}")

## 5. Truncation as desmoothing

Hewitt, Manning, and Liang (2022) read truncation as undoing smoothing. They model the LM as a mixture of a true
distribution and a near-uniform smoothing distribution (their Eq. 3), and argue for absolute and relative probability
thresholds (their §3.4) and support recovery (§4.4, App. A.2). The uniform case is exact.

**Proposition 3 (uniform desmoothing).** If $p = (1 - \lambda)p^\star + \lambda/|V|$, then
$\{w : p_w \gt \lambda/|V|\} = \operatorname{supp} p^\star$.

*Proof.* $p_w = \lambda/|V|$ exactly when $p^\star_w = 0$, and $p_w \gt \lambda/|V|$ otherwise. $\square$

A Kneser–Ney model shows its smoothing directly.

**Proposition 4 (Kneser–Ney desmoothing).** For interpolated KN,
$$p(w \mid h) = \frac{\max(c(hw) - D, 0)}{c(h)} + \gamma(h)\, p_{\text{lower}}(w \mid h^-),$$
and the seen set $\{w : c(hw) \gt 0\}$ is exactly $\{w : p(w \mid h) \gt \gamma(h)\, p_{\text{lower}}(w \mid h^-)\}$,
because $c(hw) \ge 1 \gt D$. The backoff term $\gamma(h)\, p_{\text{lower}}(w)$ varies from word to word, so no single
threshold on $p(w \mid h)$ recovers the seen set in general.

*Proof.* Read it off Topic 3's recursion. $\square$

On the 177 prompt rows whose context the trigram has seen, the seen set averages 5.34 words. On 0.847 of those rows
every seen word outranks every unseen one, so a threshold of the row's own would work. But no setting of top-$k$,
nucleus, min-$p$, $\eta$, or typical recovers the seen set on every row. Every rule trades precision against recall.

In [ ]:
st.test_uniform_desmoothing()
st.test_kn_seen_set_is_recoverable_but_no_rule_finds_it()
d = st.desmoothing_table()
for rule, grid in st.ROW_GRID.items():
    print(f"{rule:8}", " ".join(f"{x:g}:{p:.2f}/{r:.2f}" for x, p, r in zip(grid, d[rule]["precision"], d[rule]["recall"])))

## 6. Typicality

**Proposition 5 (samples are typical of the model).** Under the model,
$\mathbb{E}_p[-\log p(Y \mid x)] = \mathbb{E}_p\big[\sum_t H(p(\cdot \mid h_t))\big]$. Moreover
$M_n = \sum_{t \le n} \big(-\log p(Y_t \mid h_t) - H(h_t)\big)$ is a martingale.

*Proof.* Given $h_t$, the surprisal $-\log p(Y_t \mid h_t)$ has conditional mean $H(h_t)$. So each increment of $M_n$
has conditional mean zero, and taking expectations gives the identity. $\square$

A sample's NLL therefore tracks its own path's entropy. For i.i.d. and stationary ergodic sources this is the
asymptotic equipartition property (Cover and Thomas 2006, §3.1 and §16.8). On the trigram the pooled per-word NLLs
fall in a fixed order:
- the exact mode, 0.835;
- pure samples, 2.348;
- the held-out text, 2.576, a cross-entropy that includes the model's error.

The mode is nothing like a typical sample.

**Definition 4 (locally typical sampling).** Sort tokens by $|{-\log p_w} - H(h)|$ and keep them until their mass
reaches the typical mass (Meister et al. 2023, Alg. 5.3). At a small typical mass this drops the head of each
distribution: the likeliest token is rarely the one whose surprisal is closest to the entropy.

In [ ]:
st.test_samples_are_typical_of_the_model()
for name in ("trigram", "decoder"):
    t = st.typicality(name)
    print(f"{name:8} E[NLL − Σ H] = {t['gap']:.3f} ± {t['se']:.3f}")

## 7. Quality against diversity

Every setting gives one point per engine:
- **x axis:** the samples' per-word NLL under the generating model, which measures how far into the model's own tail
  the sampler goes;
- **y axis:** the per-word NLL under an **oracle**, a Kneser–Ney model fit to five times more text from the generator
  that wrote the training corpus (Caccia et al. 2020 use the same device).

Lower oracle NLL is more like the generator's text, but it also rewards blandness: the mode scores 1.073. Temperature
traces one curve, and the truncation rules trace nearly the same one. On this model, how much a rule cuts matters more
than which rule it is.

The held-out text is off every curve. It costs 2.576 under the trigram but only 1.968 under the oracle. Every
setting's oracle NLL exceeds its model NLL by at least 0.25 nats more than the text's does. That gap is model error,
and no decoding rule removes it.

In [ ]:
st.test_temperature_trades_quality_for_diversity()
st.test_the_text_is_off_every_curve()
st.test_draw_scores_match_the_model()
for name in ("trigram", "decoder"):
    print(name)
    for s in st.SETTINGS:
        m = st.metrics(name, s)
        print(f"  {s[0]:11} {s[1]:<7g} nll {m['nll']:.3f}  oracle {m['oracle_nll']:.3f}  distinct-2 {m['distinct2']:.3f}  "
              f"copied {m['copied']:.3f}  KL/word {m['kl']:.3f}")

## 8. Finance: boilerplate against fabrication

**Boilerplate.** Lowering $\tau$ makes samples copy training sentences more often:
- on the trigram, from 0.024 of samples at $\tau = 1.5$ to 0.148 at $\tau = 0.5$;
- on the decoder, from 0.059 to 0.226.

**Fabrication.** We probe what the models "know" with frames from the corpus's own fact templates: "Halvorsen
Industrial operates in the …", "… is classified under …", and "… is headquartered in …". #14's decoder is at chance on
these, and the trigram cannot see the company at all. An order-5 Kneser–Ney model, Topic 3's model with a longer memory,
can. We group its 111 probes by how often the 4-word context occurred in training. Every number below is an exact sum
over one next-token distribution.
- **Seen two or more times (30 probes).** The argmax is always the KB's answer. At $\tau = 1$ the model still puts 0.198
  on wrong answers, because that is the smoothing mass. As $\tau \to 0$, and as the nucleus narrows, fabrication falls
  to 0.
- **Seen once (33 probes).** A single count, discounted, competes with the backoff's sector prior. The prior wins on
  0.303 of the probes. At $\tau = 1$, P(correct) is 0.240 and P(wrong) 0.544. Lowering $\tau$ raises P(correct) to
  0.698 on average, but on the probes the prior won, fabrication goes to 1: low temperature makes the wrong answer
  certain.
- **Never seen (48 probes).** P(correct) is 0.064 at $\tau = 1$, and no temperature or truncation setting lifts it above
  0.1.

Temperature and truncation reshape what the model believes. They cannot add what it never learned.

In [ ]:
st.test_fabrication_regimes()
c = st.fabrication_curves()
i1 = st.FIN_TAUS.index(1.0)
for g in st.REGIMES:
    t = c[g]["temperature"]
    print(f"{g:7} n = {c['n'][g]:2}  argmax right {c[g]['argmax_right']:.3f}  "
          f"tau=1: correct {t['correct'][i1]:.3f} wrong {t['wrong'][i1]:.3f}  "
          f"tau={st.FIN_TAUS[0]:.3f}: correct {t['correct'][0]:.3f} wrong {t['wrong'][0]:.3f}")
for e in st.fact_examples():
    print(e["entity"], e["regime"], e["count"], e["answer"], dict(zip(e["answers"], e["p"])))

## Payload and harness

The lab's data file is `viz_payload()`, serialized compactly. The last cell runs the whole harness: every claim above,
as an assert.

In [ ]:
st.test_viz_payload()
st.test_viz_mirror_matches_payload()
print(st.VIZ_JSON.name, st.VIZ_JSON.stat().st_size, "bytes")

In [ ]:
assert st._run_harness() == 0